# Build llama-cpp-python CUDA Wheel for HF Spaces

Run this notebook **once** in a Colab GPU runtime (T4 is fine).  
It compiles a Python 3.11 + CUDA 12.x wheel and uploads it to your  
HF Dataset repo so the Dockerfile can install it in seconds instead of recompiling.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells
3. When prompted, paste a **write-access HF token**

# Build llama-cpp-python CUDA Wheel for HF Spaces

Run this notebook **once** in a Colab GPU runtime (T4 is fine).  
It compiles a Python 3.11 + CUDA 12.x wheel and uploads it to your  
HF Dataset repo so the Dockerfile can install it in seconds instead of recompiling.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells
3. When prompted, paste a **write-access HF token**

In [ ]:
# ── Cell 1: Verify GPU is available ──────────────────────────────────────────
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected. Change runtime type to T4 GPU first.')
print(result.stdout[:500])

In [ ]:
# ── Cell 2: Install Python 3.11 + build deps (Colab ships 3.10) ──────────────
!apt-get update -qq
!apt-get install -y -qq python3.11 python3.11-dev build-essential cmake ninja-build curl
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.11 -q
!python3.11 --version

In [ ]:
# ── Cell 3: Compile llama-cpp-python wheel with CUDA support ─────────────────
# This takes ~10-15 min on T4 (nvcc compiling). Only need to do this once.
import os
os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'
os.environ['FORCE_CMAKE'] = '1'

!python3.11 -m pip wheel llama-cpp-python==0.3.7 \
    --no-binary llama-cpp-python \
    --wheel-dir /tmp/llama_wheels \
    --no-deps \
    -v 2>&1 | tail -30

In [ ]:
# ── Cell 4: Confirm wheel was built and note the filename ────────────────────
import glob, os
wheels = glob.glob('/tmp/llama_wheels/*.whl')
if not wheels:
    raise RuntimeError('No wheel found — check Cell 3 output for errors.')
wheel_path = wheels[0]
wheel_name = os.path.basename(wheel_path)
print(f'Built wheel: {wheel_name}')
print(f'Size: {os.path.getsize(wheel_path) / 1e6:.1f} MB')

In [ ]:
# ── Cell 5: Upload wheel to HF Dataset repo ──────────────────────────────────
!python3.11 -m pip install -q huggingface_hub

from huggingface_hub import HfApi, login
import glob, os

# Paste a HF token with write access when prompted
login()

wheel_path = glob.glob('/tmp/llama_wheels/*.whl')[0]
wheel_name = os.path.basename(wheel_path)

REPO_ID = 'zain-0/llm-wheels'   # will be created automatically if it doesn't exist

api = HfApi()

# Create the dataset repo if it doesn't exist yet
try:
    api.create_repo(repo_id=REPO_ID, repo_type='dataset', exist_ok=True, private=False)
    print(f'Repo {REPO_ID} ready.')
except Exception as e:
    print(f'create_repo: {e}')

# Upload
url = api.upload_file(
    path_or_fileobj=wheel_path,
    path_in_repo=wheel_name,
    repo_id=REPO_ID,
    repo_type='dataset',
)
print(f'\nUploaded to: {url}')
print(f'\nDockerfile install line:')
print(f'  https://huggingface.co/datasets/{REPO_ID}/resolve/main/{wheel_name}')

In [ ]:
# Cell 1: Verify GPU is available
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected. Change runtime type to T4 GPU first.')
print(result.stdout[:500])